In [1]:
from IPython.display import HTML, display
import html as _html
import math

def htmljanje(sez):
        css = """
        <style>
        table, th, td {
          border: 1px solid white;
        }
        </style>
        """
        html = ""
        for c in sez:
            html += f"<td>{c}</td>"
        return f"{css}<table><tr>{html}</tr></table>"

In [2]:
cd = htmljanje([1,2,3,4,5,6])
display(HTML(cd))

1,2,3,4,5,6


In [3]:
class Node:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
    def _html_node_row_(self):
        return f"<hd>{self.key}</hd><td>{self.left}</td><td>{self.right}</td>"

In [ ]:
class Scapegoat:
    def __init__(self):
        self.root = None
        self.size = 0
        self.maxSize = 0
        
        
    def vstavi(self, key):
        node = Node(key)
        druzina = []
        
        #Base Case - Nothing in the tree
        if self.root == None:
            self.root = node
            return
        #Search to find the node's correct place
        currentNode = self.root
        druzina.append(currentNode)
        while currentNode != None:
            potentialParent = currentNode
            if node.key < currentNode.key:
                currentNode = currentNode.left
            else:
                currentNode = currentNode.right
                
        #Assign parents and siblings to the new node
        if node.key < potentialParent.key:
            potentialParent.left = node
        else:
            potentialParent.right = node
        node.left = None
        node.right = None
        self.size += 1
        
        return node, druzina
        
        
    def insert(self, key):
        node, druzina = self.vstavi(key)
        
        scapegoat = self.findScapegoat(node, druzina)
        if scapegoat == None:
            return
        
        tmp = self.rebalance(scapegoat)

        #Assign the correct pointers to and from scapegoat
        scapegoat.left = tmp.left
        scapegoat.right = tmp.right
        scapegoat.key = tmp.key     
        
    def findScapegoat(self, node, druzina):
        i = len(druzina)
        if i == 0:
            return None
        while self.isBalancedAtNode(node) == True:
            i -= 1
            if node == self.root:
                return None
            node = druzina[i]
        return node

    def isBalancedAtNode(self, node):
        if abs(self.sizeOfSubtree(node.left) - self.sizeOfSubtree(node.right)) <= 1:
            return True
        return False

    def sizeOfSubtree(self, node):
        if node == None:
            return 0
        return 1 + self.sizeOfSubtree(node.left) + self.sizeOfSubtree(node.right)

    def rebalance(self, root):
        def flatten(node, nodes):
            if node == None:
                return
            flatten(node.left, nodes)
            nodes.append(node)
            flatten(node.right, nodes)

        def buildTreeFromSortedList(nodes, start, end):
            if start > end:
                return None
            mid = int(math.ceil(start + (end - start) / 2.0))
            node = Node(nodes[mid].key)
            node.left = buildTreeFromSortedList(nodes, start, mid-1)
            node.right = buildTreeFromSortedList(nodes, mid+1, end)
            return node

        nodes = []
        flatten(root, nodes)
        return buildTreeFromSortedList(nodes, 0, len(nodes)-1)
    
    def display(self, node=None, scape=True):

        if self.root is None:
            display(HTML("<p>Tree is empty.</p>"))
            return

        coords = {}
        order = [0]
        max_depth = [0]

        def assign_positions(node, depth=0):
            if node is None:
                return
            assign_positions(node.left, depth + 1)
            coords[node] = (order[0], depth)
            order[0] += 1
            max_depth[0] = max(max_depth[0], depth)
            assign_positions(node.right, depth + 1)

        assign_positions(self.root)

        x_step = 80
        y_step = 90
        margin = 40
        radius = 18

        width = margin * 2 + max(1, order[0] - 1) * x_step + 2 * radius
        height = margin * 2 + max_depth[0] * y_step + 2 * radius

        def node_pixel(node):
            x_idx, depth = coords[node]
            x = margin + x_idx * x_step + radius
            y = margin + depth * y_step + radius
            return x, y

        parts = []

        def draw_edges(node):
            if node is None:
                return
            x1, y1 = node_pixel(node)
            if node.left is not None:
                x2, y2 = node_pixel(node.left)
                parts.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="#444" stroke-width="2"/>')
                draw_edges(node.left)
            if node.right is not None:
                x2, y2 = node_pixel(node.right)
                parts.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="#444" stroke-width="2"/>')
                draw_edges(node.right)

        def draw_nodes(current_node):
            if current_node is None:
                return
            x, y = node_pixel(current_node)
            label = _html.escape(str(current_node.key))
            fill_color = '#e6f2ff'
            stroke_color = '#2b6cb0'
            if node is not None and current_node is node:
                fill_color = '#ffd6d6' if scape else '#d9f7d9'
                stroke_color = '#c53030' if scape else '#2f855a'
            parts.append(f'<circle cx="{x}" cy="{y}" r="{radius}" fill="{fill_color}" stroke="{stroke_color}" stroke-width="2"/>')
            parts.append(f'<text x="{x}" y="{y + 5}" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#1a202c">{label}</text>')
            draw_nodes(current_node.left)
            draw_nodes(current_node.right)

        draw_edges(self.root)
        draw_nodes(self.root)

        svg = f"""
        <svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">
            <rect x="0" y="0" width="{width}" height="{height}" fill="white"/>
            {''.join(parts)}
        </svg>
        """
        display(HTML(svg))

    def delete(self, key):
        pass


In [17]:
drevo = Scapegoat()
drevo.vstavi(1)
drevo.vstavi(2)
noda, druzina = drevo.vstavi(3)
drevo.findScapegoat(noda, druzina)
drevo.display(noda, False)

In [ ]:
def vstavi(self, key, pokazi=False):
    node = Node(key)
    druzina = []
    
    #Base Case - Nothing in the tree
    if self.root == None:
        self.root = node
        return
    #Search to find the node's correct place
    currentNode = self.root
    druzina.append(currentNode)
    while currentNode != None:
        potentialParent = currentNode
        if node.key < currentNode.key:
            currentNode = currentNode.left
        else:
            currentNode = currentNode.right
            
    #Assign parents and siblings to the new node
    if node.key < potentialParent.key:
        potentialParent.left = node
    else:
        potentialParent.right = node
    node.left = None
    node.right = None
    self.size += 1
    
    return node, druzina